# X5 RetailHero — Raw Data Audit

This notebook establishes a reliable understanding of the raw X5 RetailHero data **before** the data is modeled in Snowflake/dbt or used for analytics and machine learning.

The audit focuses on five questions:

1. **What does each source table represent?**
2. **What is the grain and key of each table?**
3. **How do the tables relate to one another?**
4. **What data-quality issues or ambiguous fields need to be preserved and documented?**
5. **What does the source structure imply for the eventual warehouse model?**

The purpose is not to "clean everything" in pandas. Raw values are preserved; this notebook identifies issues that later transformation layers should handle explicitly.

## 1. Setup and source loading

The smaller source files are loaded fully into memory. `purchases.csv.gz` contains roughly 45.8M rows, so only a 100K-row sample is loaded for exploratory work. A separate chunked scan later validates full-table properties without loading the entire file into RAM.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

RAW = Path("../data/raw")

clients = pd.read_csv(RAW / "clients.csv.gz")
products = pd.read_csv(RAW / "products.csv.gz")
uplift_train = pd.read_csv(RAW / "uplift_train.csv.gz")
uplift_test = pd.read_csv(RAW / "uplift_test.csv.gz")

purchases_path = RAW / "purchases.csv.gz"
purchases_sample = pd.read_csv(purchases_path, nrows=100_000)

# Parse timestamps used during the audit. The raw files on disk remain unchanged.
clients["first_issue_date"] = pd.to_datetime(clients["first_issue_date"], errors="coerce")
clients["first_redeem_date"] = pd.to_datetime(clients["first_redeem_date"], errors="coerce")
purchases_sample["transaction_datetime"] = pd.to_datetime(
    purchases_sample["transaction_datetime"], errors="coerce"
)

## 2. Source inventory

A compact inventory is more useful than printing the same shape/dtype/missing-value block five times. The table below records row counts, column counts, and duplicate-row counts for the sources currently loaded.

In [ ]:
tables = {
    "clients": clients,
    "products": products,
    "uplift_train": uplift_train,
    "uplift_test": uplift_test,
    "purchases_sample": purchases_sample,
}

source_inventory = pd.DataFrame(
    {
        name: {
            "rows": len(df),
            "columns": df.shape[1],
            "duplicate_rows": df.duplicated().sum(),
        }
        for name, df in tables.items()
    }
).T

source_inventory

### Source grains expected from the raw files

- **clients** — one row per client
- **products** — one row per product
- **uplift_train** — one row per training client, with treatment and observed target
- **uplift_test** — one row per competition test client
- **purchases** — appears to be one product line per transaction; this is validated below

These are hypotheses until the key checks confirm them.

## 3. Primary keys and client partition

The first structural validation is whether the expected entity identifiers are actually unique. We also verify that `uplift_train` and `uplift_test` form a complete, non-overlapping partition of the `clients` table.

In [ ]:
key_checks = pd.DataFrame(
    [
        {
            "table": "clients",
            "rows": len(clients),
            "key": "client_id",
            "unique_keys": clients["client_id"].nunique(),
            "duplicate_keys": clients["client_id"].duplicated().sum(),
        },
        {
            "table": "products",
            "rows": len(products),
            "key": "product_id",
            "unique_keys": products["product_id"].nunique(),
            "duplicate_keys": products["product_id"].duplicated().sum(),
        },
        {
            "table": "uplift_train",
            "rows": len(uplift_train),
            "key": "client_id",
            "unique_keys": uplift_train["client_id"].nunique(),
            "duplicate_keys": uplift_train["client_id"].duplicated().sum(),
        },
        {
            "table": "uplift_test",
            "rows": len(uplift_test),
            "key": "client_id",
            "unique_keys": uplift_test["client_id"].nunique(),
            "duplicate_keys": uplift_test["client_id"].duplicated().sum(),
        },
    ]
)

display(key_checks)

client_ids = set(clients["client_id"])
product_ids = set(products["product_id"])
train_ids = set(uplift_train["client_id"])
test_ids = set(uplift_test["client_id"])

partition_check = pd.Series(
    {
        "train_test_overlap": len(train_ids & test_ids),
        "train_test_unique_clients": len(train_ids | test_ids),
        "clients_missing_from_train_test": len(client_ids - (train_ids | test_ids)),
        "train_test_ids_not_in_clients": len((train_ids | test_ids) - client_ids),
    },
    name="value",
)

partition_check

### Current finding

The key checks confirm:

- `clients.client_id` is unique across **400,162** clients.
- `products.product_id` is unique across **43,038** products.
- `uplift_train.client_id` is unique across **200,039** clients.
- `uplift_test.client_id` is unique across **200,123** clients.
- Train and test do not overlap and together cover all 400,162 clients.

This gives us clean entity keys and a clean client-level partition.

## 4. Client data quality

Client-level profiling focuses on three things that matter downstream:

- categorical domains (`gender`)
- implausible age values
- loyalty-date consistency

The goal is to identify problems, **not overwrite the raw data**. Invalid or suspicious values can later be mapped to `NULL` or flagged in the transformation layer while the original source remains intact.

In [ ]:
client_quality = {
    "gender_counts": clients["gender"].value_counts(dropna=False),
    "age_summary": clients["age"].describe(),
    "missing_first_redeem_date": clients["first_redeem_date"].isna().sum(),
}

display(client_quality["gender_counts"].to_frame("count"))
display(client_quality["age_summary"].to_frame("age"))

# Diagnostic only: inspect values outside a broadly plausible human age range.
age_outliers = clients.loc[
    ~clients["age"].between(10, 100),
    ["client_id", "age"]
].sort_values("age")

print(f"Age values outside 10–100: {len(age_outliers):,}")
display(pd.concat([age_outliers.head(10), age_outliers.tail(10)]))

redeemed = clients.loc[clients["first_redeem_date"].notna()].copy()
redeemed["redeem_delay_days"] = (
    redeemed["first_redeem_date"] - redeemed["first_issue_date"]
).dt.total_seconds() / 86_400

redeem_before_issue = redeemed["redeem_delay_days"] < 0

print(f"Missing first_redeem_date: {clients['first_redeem_date'].isna().sum():,}")
print(f"Redeem before issue: {redeem_before_issue.sum():,}")
print(f"Redeem before issue (% of redeemed clients): {redeem_before_issue.mean() * 100:.3f}%")

display(
    redeemed.loc[
        redeem_before_issue,
        ["client_id", "first_issue_date", "first_redeem_date", "redeem_delay_days"]
    ].head(10)
)

### Current finding

- `gender` contains the observed categories `U`, `F`, and `M`.
- `age` has severe impossible values (for example negative values in the thousands and values above 1,000), even though the central distribution is plausible.
- `first_redeem_date` is missing for **35,469** clients. Missing redemption dates should not automatically be treated as errors because they may mean the client had not redeemed.
- **536** redeemed clients have a redemption timestamp slightly earlier than their issue timestamp. This is a small data-quality anomaly that should be flagged rather than silently corrected.

A later cleaned customer model should preserve raw values while creating explicit validity flags or cleaned versions.

## 5. Product data quality

Product profiling checks missingness, cardinality, and known binary domains in one table. This gives us the information needed later for dbt tests and transformation rules.

In [ ]:
product_profile = pd.DataFrame({
    "dtype": products.dtypes.astype(str),
    "missing_count": products.isna().sum(),
    "missing_pct": products.isna().mean() * 100,
    "unique_values": products.nunique(dropna=True),
})

display(product_profile)

binary_domains = {
    "is_own_trademark": sorted(products["is_own_trademark"].dropna().unique().tolist()),
    "is_alcohol": sorted(products["is_alcohol"].dropna().unique().tolist()),
}

binary_domains

### Current finding

- `product_id` is complete and unique.
- `brand_id` has the most notable product-level missingness (~12%).
- `segment_id` has moderate missingness (~3.7%).
- `vendor_id` has very little missingness.
- `level_1`–`level_4` and `netto` have only a handful of missing values.
- `is_own_trademark` and `is_alcohol` both correctly use the domain `{0, 1}`.

The exact business meaning/unit of `netto` is not established strongly enough to rename it yet, so the raw name should be retained.

## 6. Uplift treatment and outcome structure

`uplift_train` supplies the client-level treatment flag and observed target. Here we validate the binary domains and inspect the treatment/control balance and raw target rates.

The observed difference is **descriptive only** at this stage. It should not be called a causal treatment effect until the treatment-assignment assumptions are established.

In [ ]:
assert set(uplift_train["treatment_flg"].unique()) <= {0, 1}
assert set(uplift_train["target"].unique()) <= {0, 1}

treatment_summary = pd.crosstab(
    uplift_train["treatment_flg"],
    uplift_train["target"],
    margins=True
)

target_rates = pd.crosstab(
    uplift_train["treatment_flg"],
    uplift_train["target"],
    normalize="index"
)

display(treatment_summary)
display(target_rates)

raw_target_rate_difference = (
    target_rates.loc[1, 1] - target_rates.loc[0, 1]
)

print(f"Raw target-rate difference: {raw_target_rate_difference:.4%}")

### Current finding

The treatment and control groups are almost perfectly balanced in size. The observed target rate is approximately:

- **Control:** 60.33%
- **Treatment:** 63.65%
- **Raw difference:** ~3.32 percentage points

This is useful as a sanity check, but no causal claim is made here.

## 7. Purchase grain, relationships, and repeated transaction-level fields

The purchase source is the most important table structurally because it mixes:

- a **transaction/basket grain** (`purchase_sum`, loyalty-point fields), and
- a **transaction-product line grain** (`product_id`, `product_quantity`, and likely other item-level fields).

The 100K-row sample is used to determine how fields behave within transactions. Exact full-table composite-key uniqueness will later be verified efficiently in Snowflake.

In [ ]:
sample_grain = pd.Series({
    "sample_rows": len(purchases_sample),
    "unique_transactions": purchases_sample["transaction_id"].nunique(),
    "unique_products": purchases_sample["product_id"].nunique(),
    "unique_transaction_product_pairs": (
        purchases_sample[["transaction_id", "product_id"]]
        .drop_duplicates()
        .shape[0]
    ),
    "duplicate_transaction_product_pairs": purchases_sample.duplicated(
        subset=["transaction_id", "product_id"]
    ).sum(),
    "max_clients_per_transaction": (
        purchases_sample.groupby("transaction_id")["client_id"].nunique().max()
    ),
    "orphan_client_ids": (
        ~purchases_sample["client_id"].isin(client_ids)
    ).sum(),
    "orphan_product_ids": (
        ~purchases_sample["product_id"].isin(product_ids)
    ).sum(),
}, name="value")

display(sample_grain.to_frame())

transaction_level_candidates = [
    "purchase_sum",
    "regular_points_received",
    "express_points_received",
    "regular_points_spent",
    "express_points_spent",
]

txn_consistency = (
    purchases_sample
    .groupby("transaction_id")[transaction_level_candidates]
    .nunique(dropna=False)
)

transaction_level_summary = pd.DataFrame({
    "pct_transactions_one_value": txn_consistency.eq(1).mean() * 100,
    "max_values_within_transaction": txn_consistency.max(),
})

display(transaction_level_summary)

### Current finding: two grains are present

In the 100K-row sample:

- `transaction_id + product_id` is unique.
- Each transaction belongs to exactly one client.
- Sampled client and product IDs have no orphan foreign keys.
- `purchase_sum` and all four loyalty-point fields are constant within **100% of sampled transactions**.

That means these fields are transaction/basket attributes repeated on every product line. Summing `purchase_sum` directly across raw purchase rows would therefore overcount revenue.

This discovery changes the candidate warehouse design: rather than one `fact_purchase`, the source naturally supports a **transaction fact** plus a **transaction-item fact**.

## 8. Ambiguous transaction-item fields

`trn_sum_from_iss` and `trn_sum_from_red` do not behave like the clearly repeated basket-level fields. We inspect their variation within transactions and whether their summed values reconcile to `purchase_sum`.

The objective is behavioral understanding only. Their precise business semantics remain unresolved and should not be invented.

In [ ]:
ambiguous_fields = ["trn_sum_from_iss", "trn_sum_from_red"]

ambiguous_consistency = (
    purchases_sample
    .groupby("transaction_id")[ambiguous_fields]
    .nunique(dropna=False)
)

display(
    pd.DataFrame({
        "pct_transactions_one_value": ambiguous_consistency.eq(1).mean() * 100,
        "max_values_within_transaction": ambiguous_consistency.max(),
    })
)

txn_amount_check = (
    purchases_sample
    .groupby("transaction_id")
    .agg(
        purchase_sum=("purchase_sum", "first"),
        trn_sum_from_iss_total=("trn_sum_from_iss", "sum"),
        trn_sum_from_red_total=("trn_sum_from_red", "sum"),
    )
)

txn_amount_check["combined_trn_sum"] = (
    txn_amount_check["trn_sum_from_iss_total"]
    + txn_amount_check["trn_sum_from_red_total"]
)

txn_amount_check["difference_from_purchase_sum"] = (
    txn_amount_check["combined_trn_sum"]
    - txn_amount_check["purchase_sum"]
)

display(txn_amount_check.head(10))
display(txn_amount_check["difference_from_purchase_sum"].describe().to_frame())

### Current finding

- `trn_sum_from_iss` varies within most transactions, which is more consistent with an item-level measure than a basket-level measure.
- `trn_sum_from_red` appears constant more often, but this is heavily influenced by its very high missingness.
- `trn_sum_from_iss + trn_sum_from_red` does **not** reliably reconcile to `purchase_sum`.

Therefore these columns should retain their original names and be documented as semantically unresolved until an authoritative definition is found.

## 9. Full purchase-table audit (chunked)

The full purchase source is too large to load comfortably into pandas all at once. A chunked scan validates properties that can be accumulated safely:

- total row count
- full date range
- missing-value counts
- client/product referential integrity
- timestamp parseability

Global `transaction_id + product_id` uniqueness will be checked later in Snowflake, which is a better system for a 45M-row group-by.

In [ ]:
CHUNK_SIZE = 500_000

total_rows = 0
missing_counts = None
full_min_date = None
full_max_date = None
missing_client_fk = 0
missing_product_fk = 0
invalid_datetime_count = 0

for chunk in pd.read_csv(purchases_path, chunksize=CHUNK_SIZE):
    total_rows += len(chunk)

    chunk_missing = chunk.isna().sum()
    missing_counts = (
        chunk_missing.copy()
        if missing_counts is None
        else missing_counts.add(chunk_missing, fill_value=0)
    )

    missing_client_fk += (~chunk["client_id"].isin(client_ids)).sum()
    missing_product_fk += (~chunk["product_id"].isin(product_ids)).sum()

    dates = pd.to_datetime(chunk["transaction_datetime"], errors="coerce")
    invalid_datetime_count += dates.isna().sum()

    chunk_min = dates.min()
    chunk_max = dates.max()

    full_min_date = chunk_min if full_min_date is None else min(full_min_date, chunk_min)
    full_max_date = chunk_max if full_max_date is None else max(full_max_date, chunk_max)

full_purchase_audit = pd.Series({
    "total_rows": total_rows,
    "min_transaction_datetime": full_min_date,
    "max_transaction_datetime": full_max_date,
    "orphan_client_ids": int(missing_client_fk),
    "orphan_product_ids": int(missing_product_fk),
    "invalid_timestamps": int(invalid_datetime_count),
}, name="value")

display(full_purchase_audit.to_frame())
display(
    missing_counts.astype("int64")
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

### Current full-table findings

The completed full scan found:

- **45,786,568** purchase rows
- transaction timestamps from **2018-11-21 21:02:33** through **2019-03-18 23:40:03**
- **0** purchase client IDs missing from `clients`
- **0** purchase product IDs missing from `products`
- **0** invalid transaction timestamps
- `trn_sum_from_red` has **42,743,212** missing values; the other purchase fields have no observed missing values

This confirms that the purchase source is structurally very clean except for the highly sparse `trn_sum_from_red` field.

## 10. Warehouse implications

The raw purchase source contains two different logical grains, so the initial warehouse model should separate them.

### Candidate foundation model

```text
                         dim_customer
                              |
                 +------------+-------------+
                 |                          |
                 v                          v
          fact_transaction       fact_treatment_outcome
                 |
                 | transaction_id
                 v
       fact_transaction_item
                 |
                 | product_id
                 v
             dim_product
```

### Why this design

**`dim_customer`**  
One row per client. Describes who the customer is.

**`dim_product`**  
One row per product. Describes what the product is.

**`fact_transaction`**  
One row per transaction/basket. This is where repeated transaction-level fields such as `purchase_sum` and loyalty-point totals belong.

**`fact_transaction_item`**  
One row per transaction-product pair. This stores the product-level composition of each basket, including quantity and item-level measures.

**`fact_treatment_outcome`**  
One row per uplift-training client containing treatment and observed target.

`dim_date` and `dim_store` remain candidates rather than commitments until their usefulness is evaluated.

## 11. Audit conclusions and deferred decisions

### Confirmed

- Client, product, uplift-train, and uplift-test entity keys are unique.
- Train and test form a complete non-overlapping partition of the client population.
- Purchase client/product foreign keys are valid across the full 45.8M-row source.
- Purchase timestamps are fully parseable.
- The raw purchase source mixes transaction-level and transaction-item-level fields.
- `purchase_sum` and loyalty-point measures are transaction-level values repeated on line items.
- Product binary indicators have valid `{0, 1}` domains.
- Client age contains obvious invalid values.
- A small number of redemption timestamps precede issue timestamps.
- `trn_sum_from_red` is highly sparse.

### Deliberately unresolved

- Exact semantic definitions of `netto`, `trn_sum_from_iss`, and `trn_sum_from_red`
- Final cleaning treatment for invalid ages and anomalous dates
- Final warehouse treatment of store/date dimensions
- Full-table uniqueness of `(transaction_id, product_id)` — to be verified in Snowflake
- Causal interpretation of the uplift treatment — treatment assignment assumptions must be established before making causal claims

### Next step

Move the raw sources into **Snowflake** and create a faithful raw layer. The next SQL-based work will:

1. validate the full purchase grain,
2. reproduce source row counts and key checks in the warehouse,
3. perform business-level exploration over all 45.8M purchase rows, and
4. prepare for dbt-based dimensional modeling.